# Coalesced access with double precision

This notebook adapts `simpleExampleDouble.cu`. Its purpose is to give a concrete coalesced baseline, not to measure a speedup: each lane reads consecutive `double` elements and writes a consecutive result. The program prints the lane-to-index mapping and the corresponding sector count, then checks the full result against the CPU and reports a kernel time.


## Select a GPU runtime

In Colab, choose **Runtime → Change runtime type → GPU**. This cell shows the GPU and CUDA compiler available to the notebook.

In [1]:
!nvidia-smi
!nvcc --version
!mkdir -p code


Sat Nov 14 14:59:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## CUDA error and timing support

`CUDA_CHECK` handles CUDA errors independently of `TimingGPU`. The timer uses CUDA events and reports elapsed milliseconds for GPU work.

In [2]:
%%writefile code/cuda_check.cuh
#ifndef CUDA_CHECK_CUH
#define CUDA_CHECK_CUH
#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>
#define CUDA_CHECK(call) do { \
    cudaError_t status = (call); \
    if (status != cudaSuccess) { \
        std::fprintf(stderr, "%s:%d: %s\n", __FILE__, __LINE__, \
                     cudaGetErrorString(status)); \
        std::exit(EXIT_FAILURE); \
    } \
} while (0)
#endif


Writing code/cuda_check.cuh


In [3]:
%%writefile code/TimingGPU.cuh
#ifndef TIMING_GPU_CUH
#define TIMING_GPU_CUH
#include <cuda_runtime.h>
class TimingGPU {
    cudaEvent_t start_;
    cudaEvent_t stop_;
    bool started_;
public:
    TimingGPU();
    ~TimingGPU();
    TimingGPU(const TimingGPU&) = delete;
    TimingGPU& operator=(const TimingGPU&) = delete;
    void StartCounter();
    float GetCounter(); // milliseconds
};
#endif


Writing code/TimingGPU.cuh


In [4]:
%%writefile code/TimingGPU.cu
#include "TimingGPU.cuh"
#include "cuda_check.cuh"
#include <cstdio>
#include <cstdlib>

TimingGPU::TimingGPU() : started_(false) {
    CUDA_CHECK(cudaEventCreate(&start_));
    CUDA_CHECK(cudaEventCreate(&stop_));
}
TimingGPU::~TimingGPU() {
    CUDA_CHECK(cudaEventDestroy(start_));
    CUDA_CHECK(cudaEventDestroy(stop_));
}
void TimingGPU::StartCounter() {
    CUDA_CHECK(cudaEventRecord(start_));
    started_ = true;
}
float TimingGPU::GetCounter() {
    if (!started_) {
        std::fprintf(stderr, "Call StartCounter before GetCounter.\n");
        std::exit(EXIT_FAILURE);
    }
    CUDA_CHECK(cudaEventRecord(stop_));
    CUDA_CHECK(cudaEventSynchronize(stop_));
    float milliseconds = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&milliseconds, start_, stop_));
    started_ = false;
    return milliseconds;
}


Writing code/TimingGPU.cu


## CUDA kernel and host program

Three consecutive lanes access three consecutive 8-byte elements in each array. `cudaMalloc` provides an aligned base, and the launch uses full warps. This is a coalesced baseline, not a comparison of float and double arithmetic.

In [5]:
%%writefile code/chapter02_coalesced_double.cu
#include "cuda_check.cuh"
#include "TimingGPU.cuh"
#include <cstdio>
#include <cstdlib>

const int N = 1 << 20;
const int THREADS = 256;
const int REPEATS = 100;

__global__ void add_double(const double *a, const double *b,
                           double *c, int n) {
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

void launch_add(const double *a, const double *b, double *c, int n) {
    const int blocks = (n + THREADS - 1) / THREADS;
    add_double<<<blocks, THREADS>>>(a, b, c, n);
    CUDA_CHECK(cudaGetLastError());
}

int main() {
    const size_t bytes = size_t(N) * sizeof(double);
    double *h_a = (double*)std::malloc(bytes);
    double *h_b = (double*)std::malloc(bytes);
    double *h_cpu = (double*)std::malloc(bytes);
    double *h_gpu = (double*)std::malloc(bytes);
    double *d_a = nullptr, *d_b = nullptr, *d_c = nullptr;
    if (!h_a || !h_b || !h_cpu || !h_gpu) {
        std::fprintf(stderr, "Host allocation failed.\n");
        return EXIT_FAILURE;
    }
    for (int i = 0; i < N; ++i) {
        h_a[i] = double(i % 97);
        h_b[i] = double((i * 3) % 101);
        h_cpu[i] = h_a[i] + h_b[i];
    }
    CUDA_CHECK(cudaMalloc((void**)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void**)&d_c, bytes));
    CUDA_CHECK(cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice));

    launch_add(d_a, d_b, d_c, N);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(h_gpu, d_c, bytes, cudaMemcpyDeviceToHost));
    for (int i = 0; i < N; ++i) {
        if (h_gpu[i] != h_cpu[i]) {
            std::fprintf(stderr, "CPU/GPU mismatch at i=%d.\n", i);
            return EXIT_FAILURE;
        }
    }
    std::printf("CPU vs GPU: PASS (%d double values)\n", N);

    std::printf("Warp mapping (first four lanes):\n");
    for (int lane = 0; lane < 4; ++lane) {
        const int first_byte = lane * int(sizeof(double));
        const int last_byte = first_byte + int(sizeof(double)) - 1;
        std::printf("  lane %d: read a[%d], b[%d], write c[%d]; byte offsets %d--%d\n",
                    lane, lane, lane, lane, first_byte, last_byte);
    }
    const int warp_lanes = 32;
    const int sector_bytes = 32;
    const int bytes_per_warp = warp_lanes * int(sizeof(double));
    const int sectors_per_instruction =
        (bytes_per_warp + sector_bytes - 1) / sector_bytes;
    std::printf("A warp requests %d contiguous bytes from each array.\n", bytes_per_warp);
    std::printf("With aligned bases and 32-byte sectors, each load/store instruction covers %d sectors.\n",
                sectors_per_instruction);
    std::printf("The printed time is a baseline measurement, not a coalesced-versus-uncoalesced comparison.\n");

    for (int k = 0; k < 10; ++k) launch_add(d_a, d_b, d_c, N);
    CUDA_CHECK(cudaDeviceSynchronize());
    TimingGPU timer;
    timer.StartCounter();
    for (int k = 0; k < REPEATS; ++k) launch_add(d_a, d_b, d_c, N);
    const float ms = timer.GetCounter() / REPEATS;
    std::printf("Aligned double vector addition: %.6f ms/launch (average of %d)\n",
                ms, REPEATS);

    CUDA_CHECK(cudaFree(d_a)); CUDA_CHECK(cudaFree(d_b)); CUDA_CHECK(cudaFree(d_c));
    std::free(h_a); std::free(h_b); std::free(h_cpu); std::free(h_gpu);
    return EXIT_SUCCESS;
}


Writing code/chapter02_coalesced_double.cu


## Compile and run

The program prints the address pattern of the first lanes and the number of 32-byte sectors for one warp, then validates the full GPU result against the CPU and times repeated launches. The timing excludes allocation, host-device copies, and validation. It measures this kernel on the selected GPU; by itself, it does not quantify a speedup from coalescing.


In [6]:
%%bash
set -eu
nvcc -O2 -std=c++14 -Icode code/chapter02_coalesced_double.cu code/TimingGPU.cu -o memory_example
./memory_example


CPU vs GPU: PASS (1048576 double values)
Warp mapping (first four lanes):
  lane 0: read a[0], b[0], write c[0]; byte offsets 0--7
  lane 1: read a[1], b[1], write c[1]; byte offsets 8--15
  lane 2: read a[2], b[2], write c[2]; byte offsets 16--23
  lane 3: read a[3], b[3], write c[3]; byte offsets 24--31
A warp requests 256 contiguous bytes from each array.
With aligned bases and 32-byte sectors, each load/store instruction covers 8 sectors.
The printed time is a baseline measurement, not a coalesced-versus-uncoalesced comparison.
Aligned double vector addition: 0.100977 ms/launch (average of 100)
